In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from PIL import Image

In [2]:
device = torch.device("cpu")

In [3]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [4]:
CAR_TRAIN = "Cars Dataset/train"
DAMAGE_TRAIN = "Car_Damage/train"

car_data = datasets.ImageFolder(CAR_TRAIN, transform=transform)
damage_data = datasets.ImageFolder(DAMAGE_TRAIN, transform=transform)

car_loader = torch.utils.data.DataLoader(car_data, batch_size=8, shuffle=True)
damage_loader = torch.utils.data.DataLoader(damage_data, batch_size=8, shuffle=True)

car_classes = car_data.classes
print("Car Classes:", car_classes)

Car Classes: ['Audi', 'Hyundai Creta', 'Mahindra Scorpio', 'Rolls Royce', 'Swift', 'Tata Safari', 'Toyota Innova']


In [5]:
class MultiTaskModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        base = models.mobilenet_v2(pretrained=True)

        # Freeze backbone
        for param in base.parameters():
            param.requires_grad = False

        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)

        # Brand head
        self.brand = nn.Linear(1280, num_classes)

        # Damage head
        self.damage = nn.Linear(1280, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)

        return self.brand(x), self.damage(x)

In [6]:
model = MultiTaskModel(len(car_classes)).to(device)

criterion_brand = nn.CrossEntropyLoss()
criterion_damage = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0003)

C:\Users\SONY\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\SONY\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
print("\nTraining...")

for epoch in range(5): 
    model.train()
    total_loss = 0

    for (car_img, car_label), (dam_img, dam_label) in zip(car_loader, damage_loader):

        car_img, car_label = car_img.to(device), car_label.to(device)
        dam_img = dam_img.to(device)
        dam_label = dam_label.float().unsqueeze(1).to(device)

        brand_out, _ = model(car_img)
        _, damage_out = model(dam_img)

        loss1 = criterion_brand(brand_out, car_label)
        loss2 = criterion_damage(damage_out, dam_label)

        loss = loss1 + loss2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Training...
Epoch 1, Loss: 184.3088
Epoch 2, Loss: 153.1149
Epoch 3, Loss: 136.1424
Epoch 4, Loss: 128.9530
Epoch 5, Loss: 121.3132


In [8]:
torch.save(model.state_dict(), "final_model.pth")
print("Model saved!")

Model saved!


In [9]:
def predict(image_path):
    model.eval()

    img = Image.open(image_path).convert('RGB')
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        brand_out, damage_out = model(img)

        brand = car_classes[torch.argmax(brand_out).item()]
        damage = "Damaged" if torch.sigmoid(damage_out).item() > 0.5 else "Not Damaged"

    print("\nPrediction:")
    print("Brand:", brand)
    print("Damage:", damage)

In [22]:
predict("Cars Dataset/test/Mahindra Scorpio/164.jpg")


Prediction:
Brand: Mahindra Scorpio
Damage: Not Damaged
